# Sales Forecasting & Business Performance Dashboard
## Phase 1 & 2 — Data Cleaning & Exploratory Data Analysis (EDA)

**Tools Used:** Python, pandas, numpy, matplotlib, seaborn  
**Dataset:** Sample Superstore Sales Dataset (Kaggle)  
**Goal:** Clean raw sales data and extract key business insights through visualizations

---

### Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

OUTPUT_DIR = r'D:\Sales-Forecast-Dashboard\outputs'
DATA_PATH  = r'D:\Sales-Forecast-Dashboard\data\superstore_raw.csv'
CLEAN_PATH = r'D:\Sales-Forecast-Dashboard\data\superstore_cleaned.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Libraries loaded successfully!')

---
### Step 2 — Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='windows-1252')
print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

In [ ]:
# Column data types
df.dtypes

---
### Step 3 — Data Cleaning

In [ ]:
# Parse date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=False)

# Remove duplicates
before = len(df)
df.drop_duplicates(subset='Row ID', keep='first', inplace=True)
print(f'Duplicates removed: {before - len(df)}')
print(f'Final dataset size: {len(df)} rows')

# Check for null values
null_counts = df.isnull().sum()
print(f'\nNull values: {null_counts[null_counts > 0].to_dict() or "None found"}')

In [ ]:
# Add useful time-based columns
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
df['YearMonth']  = df['Order Date'].dt.to_period('M')

# Save cleaned dataset
df.to_csv(CLEAN_PATH, index=False)
print(f'Cleaned dataset saved -> {CLEAN_PATH}')
df[['Order Date','Year','Month','Month_Name','YearMonth']].head()

---
### Step 4 — EDA: Chart 1 — Monthly Sales Trend
> **Insight:** This shows how sales grow month over month — reveals seasonality patterns

In [ ]:
monthly = (df.groupby('YearMonth')['Sales']
             .sum()
             .reset_index()
             .rename(columns={'Sales': 'Total_Sales'}))
monthly['YearMonth'] = monthly['YearMonth'].astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly['YearMonth'], monthly['Total_Sales'], marker='o', color='#1a6faf', linewidth=2)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Monthly Sales Trend (2014-2017)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_monthly_sales_trend.png', dpi=150)
plt.show()

---
### Step 5 — EDA: Chart 2 — Region-wise Sales
> **Insight:** Which region contributes the most to total revenue?

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(region_sales.index, region_sales.values,
               color=['#1a6faf','#2d9cdb','#56ccf2','#bde0fe'])
plt.title('Region-wise Total Sales', fontsize=14, fontweight='bold')
plt.ylabel('Total Sales ($)')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_region_wise_sales.png', dpi=150)
plt.show()

---
### Step 6 — EDA: Chart 3 — Category Sales vs Profit
> **Insight:** Not all high-sales categories are profitable — Technology leads in both

In [ ]:
cat = df.groupby('Category')[['Sales','Profit']].sum().reset_index()

x = np.arange(len(cat))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, cat['Sales'],  width, label='Sales',  color='#1a6faf')
ax.bar(x + width/2, cat['Profit'], width, label='Profit', color='#56ccf2')
ax.set_xticks(x)
ax.set_xticklabels(cat['Category'], fontsize=11)
ax.set_title('Category-wise Sales vs Profit', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_category_sales_profit.png', dpi=150)
plt.show()

---
### Step 7 — EDA: Chart 4 — Top 10 Products by Revenue
> **Insight:** Focus marketing efforts on these top-performing products

In [ ]:
top10 = df.groupby('Product Name')['Sales'].sum().nlargest(10).sort_values()

plt.figure(figsize=(12, 6))
bars = plt.barh(top10.index, top10.values, color='#1a6faf')
plt.title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Total Sales ($)')
for bar in bars:
    plt.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
             f'${bar.get_width():,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_top10_products.png', dpi=150)
plt.show()

---
### Step 8 — EDA: Chart 5 — Discount vs Profit
> **Insight:** High discounts (>40%) consistently result in negative profit — a key business problem

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df['Discount'], df['Profit'], alpha=0.4, color='#1a6faf', edgecolors='none')
plt.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Break-even line')
plt.title('Discount vs Profit  —  Does more discount hurt profits?', fontsize=13, fontweight='bold')
plt.xlabel('Discount Rate')
plt.ylabel('Profit ($)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_discount_vs_profit.png', dpi=150)
plt.show()

---
### Step 9 — EDA: Chart 6 — Year-over-Year Sales Comparison
> **Insight:** Sales grow each year — compare seasonal patterns across 2014-2017

In [ ]:
yoy   = df.groupby(['Year','Month'])['Sales'].sum().reset_index()
pivot = yoy.pivot(index='Month', columns='Year', values='Sales')

plt.figure(figsize=(12, 5))
colors = ['#1a6faf','#2d9cdb','#56ccf2','#90e0ef']
for i, yr in enumerate(pivot.columns):
    plt.plot(pivot.index, pivot[yr], marker='o', label=str(yr),
             linewidth=2, color=colors[i])
plt.xticks(range(1,13), ['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
plt.title('Year-over-Year Monthly Sales Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Total Sales ($)')
plt.legend(title='Year')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_yoy_sales_comparison.png', dpi=150)
plt.show()

---
### Step 10 — EDA: Chart 7 — Correlation Heatmap
> **Insight:** Understand relationships between Sales, Profit, Discount and Quantity

In [ ]:
corr_cols = ['Sales','Profit','Discount','Quantity']
corr = df[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_correlation_heatmap.png', dpi=150)
plt.show()

---
### Step 11 — EDA: Chart 8 — Sales by Customer Segment
> **Insight:** Which customer segment drives the most revenue?

In [ ]:
seg = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(7, 5))
plt.pie(seg.values, labels=seg.index, autopct='%1.1f%%',
        colors=['#1a6faf','#56ccf2','#bde0fe'], startangle=140,
        textprops={'fontsize': 12})
plt.title('Sales by Customer Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_segment_sales_pie.png', dpi=150)
plt.show()

---
### Step 12 — Key Business Insights Summary

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
profit_margin = (total_profit / total_sales) * 100
best_region   = df.groupby('Region')['Sales'].sum().idxmax()
best_month    = monthly.loc[monthly['Total_Sales'].idxmax(), 'YearMonth']
best_category = df.groupby('Category')['Profit'].sum().idxmax()

print('=' * 50)
print('       KEY BUSINESS INSIGHTS SUMMARY')
print('=' * 50)
print(f'  Total Revenue     : ${total_sales:,.2f}')
print(f'  Total Profit      : ${total_profit:,.2f}')
print(f'  Overall Margin    : {profit_margin:.1f}%')
print(f'  Best Region       : {best_region}')
print(f'  Peak Sales Month  : {best_month}')
print(f'  Most Profitable   : {best_category} category')
print(f'  Max Discount Given: {df["Discount"].max()*100:.0f}%')
print('=' * 50)
print()
print('Business Findings:')
print('  1. High discounts (>40%) consistently result in LOSSES')
print('  2. Technology is the most profitable category')
print('  3. Sales peak every Nov-Dec (festive/end-of-year season)')
print('  4. West region leads in total revenue')
print('  5. Consumer segment contributes ~50% of total sales')
print()
print(f'All 8 charts saved to: {OUTPUT_DIR}')